In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('data.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  str    
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  str    
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), str(2)
memory usage: 8.0 MB


In [4]:
X=df.drop(columns=['diabetes'])
y=df[['diabetes']]

In [5]:
X = pd.get_dummies(X, drop_first=True)

In [6]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [7]:
from imblearn.over_sampling import SMOTE
smote=SMOTE(random_state=42)
x_resampled,y_resampled=smote.fit_resample(X_train,y_train)

In [8]:
print(y_train['diabetes'].value_counts())
print(y_test['diabetes'].value_counts())

diabetes
0    73200
1     6800
Name: count, dtype: int64
diabetes
0    18300
1     1700
Name: count, dtype: int64


In [9]:
y_resampled['diabetes'].value_counts()

diabetes
1    73200
0    73200
Name: count, dtype: int64

In [10]:
#build model and train - RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
model_rf = RandomForestClassifier(n_estimators=50, random_state=42)
model_rf.fit(x_resampled, y_resampled)

#evaluate model
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = model_rf.predict(X_test)
y_pred = pd.DataFrame(data = y_pred, columns = ['Prediction'])
result = pd.concat([y_test.reset_index(drop = True), y_pred], axis = 1)
cm = confusion_matrix(result['diabetes'], result['Prediction'])
score = accuracy_score(result['diabetes'], result['Prediction'])
print(cm)
print(score)

[[17936   364]
 [  448  1252]]
0.9594


In [11]:
import pickle

pickle.dump(model_rf, open('model.pkl', 'wb'))
pickle.dump(X.columns.tolist(), open("columns.pkl", "wb"))

In [12]:
columns = [
    'gender', 'age', 'hypertension', 'heart_disease',
    'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level'
]

# 1. Input dataframe
myinput = pd.DataFrame(
    [['Male', 65, 1, 1, 'former', 35.0, 8.5, 220]],
    columns=columns
)

# 2. One-hot encoding
myinput = pd.get_dummies(
    myinput,
    columns=['gender', 'smoking_history'],
    drop_first=True
)

# 3. Align columns with training data
myinput = myinput.reindex(columns=X.columns, fill_value=0)

# 4. Predict (NO SCALING NEEDED)
result = model_rf.predict(myinput)

# 5. Output
if result[0] == 0:
    print("No, patient is not diabetic")
else:
    print("Yes, patient is diabetic")

Yes, patient is diabetic
